# ML Assignment 02
### Dataset: House Price Prediction Dataset
### Total Marks: 100

---

## Exam Instructions:
1. প্রথমে নিচের cell এ নিজের **নাম** এবং কোর্সে registration করা **ইমেইল** দিবে
2. Question wise numbering করে Text cell রাখবে এবং এর নিচে Code cell থাকবে, চেষ্টা করবে একটি code cell এ একটি question উত্তর দেওয়ার
3. Google Colab এর মধ্যে কোডগুলো করবে
4. এবং সেই ফাইলটি **'Anyone with the link' & 'View' Access** দিয়ে ফাইলটির Shareable Link টি সাবমিট করবে

---

**Question Dataset Link:** https://www.kaggle.com/datasets/prokshitha/home-value-insights

## Student Information

In [248]:
# Fill in your information
name = "Muktadir Haque Sarker"           # Write your full name here
email = "***********@gmail.com"          # Write your registered email here

print(f"Name  : {name}")
print(f"Email : {email}")

Name  : Muktadir Haque Sarker
Email : ***********@gmail.com


In [249]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import SGDRegressor, LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, root_mean_squared_error

import kagglehub
from kagglehub import KaggleDatasetAdapter

---
## Question 1 (10 Marks)

Load the House Price dataset and display:
- Dataset shape
- First 10 rows
- 5 random samples

In [250]:
# Question 1
df = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS, "prokshitha/home-value-insights", "house_price_regression_dataset.csv")
print("\n", df.shape)
display(df.head(10))
display(df.sample(5))

Using Colab cache for faster access to the 'home-value-insights' dataset.

 (1000, 8)


,Square_Footage,Num_Bedrooms,Num_Bathrooms,Year_Built,Lot_Size,Garage_Size,Neighborhood_Quality,House_Price
0,1360,2,1,1981,0.599637,0,5,2.623829e+05
1,4272,3,3,2016,4.753014,1,6,9.852609e+05
2,3592,1,2,2016,3.634823,0,9,7.779774e+05
3,966,1,2,1977,2.730667,1,8,2.296989e+05
4,4926,2,1,1993,4.699073,0,8,1.041741e+06
5,3944,5,3,1990,2.475930,2,8,8.797970e+05
6,3671,1,2,2012,4.911960,0,1,8.144279e+05
7,3419,1,1,1972,2.805281,1,1,7.034131e+05
8,630,3,3,1997,1.014286,1,8,1.738750e+05
9,2185,4,2,1981,3.941604,2,5,5.041765e+05


,Square_Footage,Num_Bedrooms,Num_Bathrooms,Year_Built,Lot_Size,Garage_Size,Neighborhood_Quality,House_Price
321,4642,4,1,1966,0.702326,1,10,9.154412e+05
652,2253,3,2,2012,3.956577,1,4,5.514446e+05
888,4521,5,3,2008,4.562175,2,2,1.039730e+06
205,4999,5,1,1952,4.662712,2,5,1.060976e+06
711,2105,3,1,2021,0.543591,1,10,4.722920e+05


---
## Question 2 (10 Marks)

Handle missing values and perform feature engineering:
- Impute missing numerical values using `SimpleImputer` with mean strategy
- Impute missing categorical values using most frequent strategy
- Drop columns with more than 50% missing values
- Perform train-test split with `test_size=0.2` and `random_state=42`

Display the shape of final train and test sets.

In [251]:
#Question 2

thresh_limit = len(df) * 0.50
df = df.dropna(thresh=thresh_limit, axis=1)

X = df.drop('House_Price', axis=1)
y = df['House_Price']

numeric_features = df.select_dtypes(include=["int64", "float64"]).columns.delete(-1)
categorical_features = df.select_dtypes(include=['object', 'category']).columns


num_imputer = SimpleImputer(strategy='mean')
cat_imputer = SimpleImputer(strategy='most_frequent')


num_imputed = num_imputer.fit_transform(X[numeric_features])
df_num = pd.DataFrame(num_imputed, columns=numeric_features)
if len(categorical_features) > 0:
    cat_imputed = cat_imputer.fit_transform(X[categorical_features])
    df_cat = pd.DataFrame(cat_imputed, columns=categorical_features)

    X_imputed = pd.concat([df_num, df_cat], axis=1)

else:
    X_imputed = df_num



X_train, X_test, y_train, y_test = train_test_split(X_imputed, y, test_size=0.2, random_state=42)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(800, 7)
(200, 7)
(800,)
(200,)


---
## Question 3 (20 Marks)

Implement **Simple Linear Regression** using **only NumPy** (no Scikit-Learn allowed):
- Compute slope (`m`) and intercept (`c`) using the Batch Gradient Descent
- Predict values for the test set
- Print the learned `m` and `c` values

Use `Square_Footage` as feature (X) and `House_Price` as target (y).

In [252]:
#Question 3


X_train_temp = X_train['Square_Footage'].values
y_train_temp = y_train.values
X_test_temp = X_test['Square_Footage'].values


X_mean = np.mean(X_train_temp)
X_std = np.std(X_train_temp)
X_train_scaled = (X_train_temp - X_mean) / X_std


m_scaled = 0.0
c_scaled = 0.0
learning_rate = 0.01
epochs = 1000            # max_iter
n_samples = len(X_train_scaled)

for _ in range(epochs):
    y_pred = m_scaled * X_train_scaled + c_scaled
    errors = y_pred - y_train_temp

    dj_dm = (1/n_samples) * np.dot(errors, X_train_scaled)
    dj_dc = (1/n_samples) * np.sum(errors)

    m_scaled -= learning_rate * dj_dm
    c_scaled -= learning_rate * dj_dc

m_learned = m_scaled / X_std
c_learned = c_scaled - (m_scaled * X_mean / X_std)


def predict(X, m, c):
    return (m * X) + c


print(f"\nLearned slope (m): {m_learned:.4f}")
print(f"Learned intercept (c): {c_learned:.4f}\n")

# Predict values for the test set
y_test_pred = predict(X_test_temp, m_learned, c_learned)

print("First 5 Test Predictions:")
print(y_test_pred[:5])



Learned slope (m): 200.5482
Learned intercept (c): 54226.7175

First 5 Test Predictions:
[ 858826.17040054  517493.10237831  998407.73053184 1043330.53149364
  785425.52240046]


---
## Question 4 (10 Marks)

Build a **ColumnTransformer** that applies:
- `StandardScaler` on numerical columns: `Square_Footage`, `Num_Bedrooms`, `Num_Bathrooms`
- `OneHotEncoder` on categorical column: `Neighborhood_Quality`



In [253]:
#Question 4

numeric_cols = ['Square_Footage', 'Num_Bedrooms', 'Num_Bathrooms']
categorical_cols = ['Neighborhood_Quality']

X_train['Neighborhood_Quality'] = X_train['Neighborhood_Quality'].astype(str)
X_test['Neighborhood_Quality'] = X_test['Neighborhood_Quality'].astype(str)

col_transformer = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ],

    remainder='passthrough'
)

col_transformer

ColumnTransformer(remainder='passthrough',
                  transformers=[('num', StandardScaler(),
                                 ['Square_Footage', 'Num_Bedrooms',
                                  'Num_Bathrooms']),
                                ('cat',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['Neighborhood_Quality'])])

## Question 5 (20 Marks)

Build a complete **Pipeline** using Scikit-Learn that includes:
- The `ColumnTransformer`
- `SGDRegressor` as the final estimator
- Train the pipeline and evaluate using RMSE and R² score
- Print predicted vs actual values for the first 10 test samples

In [254]:
# Question 5

numeric_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns


col_transformer = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ],

    remainder='passthrough'
)


sgd_pipeline = Pipeline(
    steps=[
        ("preprocessor", col_transformer),
        ('model', SGDRegressor(learning_rate="adaptive"))
    ]
)
sgd_pipeline.fit(X_train, y_train)

y_sgd_pred = sgd_pipeline.predict(X_test)

rmse_sgd = root_mean_squared_error(y_test, y_sgd_pred)
r2_sgd = r2_score(y_test, y_sgd_pred)

print(f"SGDRegressor RMSE: {rmse_sgd:.4f}")
print(f"SGDRegressor R² Score: {r2_sgd:.4f}\n")

# Display first 10 predicted vs actual
results_sgd = pd.DataFrame({'Actual': y_test[:10].values, 'Predicted': y_sgd_pred[:10]})
print("First 10 Samples (Predicted vs Actual):")
display(results_sgd)

SGDRegressor RMSE: 10253.2817
SGDRegressor R² Score: 0.9984

First 10 Samples (Predicted vs Actual):


,Actual,Predicted
0,9.010005e+05,8.698510e+05
1,4.945375e+05,4.930352e+05
2,9.494042e+05,9.435271e+05
3,1.040389e+06,1.032302e+06
4,7.940100e+05,7.761692e+05
5,7.240336e+05,7.330173e+05
6,9.984392e+05,9.923488e+05
7,9.097134e+05,8.857155e+05
8,7.926815e+05,7.961981e+05
9,9.474908e+05,9.320092e+05


---
## Question 6 (20 Marks)

Implement **Multiple Linear Regression** using **Scikit-Learn**:
- The `ColumnTransformer`
- `LinearRegression` as the final estimator
- Train the pipeline and evaluate using RMSE and R² score
- Print predicted vs actual values for the first 10 test samples

In [255]:
# Question 6

lr_pipeline = Pipeline(
    steps=[
        ('preprocessor', col_transformer),
        ('model', LinearRegression())
    ]
)

lr_pipeline.fit(X_train, y_train)
y_lr_pred = lr_pipeline.predict(X_test)


rmse_lr = root_mean_squared_error(y_test, y_lr_pred)
r2_lr = r2_score(y_test, y_lr_pred)

print(f"Linear Regression RMSE: {rmse_lr:.4f}")
print(f"Linear Regression R² Score: {r2_lr:.4f}\n")

results_lr = pd.DataFrame(
    {
        'Actual': y_test[:10],
        'Predicted': y_lr_pred[:10]
    }
)

print("First 10 Samples (Predicted vs Actual):")
display(results_lr)

Linear Regression RMSE: 10250.3104
Linear Regression R² Score: 0.9984

First 10 Samples (Predicted vs Actual):


,Actual,Predicted
521,9.010005e+05,8.698744e+05
737,4.945375e+05,4.930080e+05
740,9.494042e+05,9.435438e+05
660,1.040389e+06,1.032332e+06
411,7.940100e+05,7.761808e+05
678,7.240336e+05,7.330118e+05
626,9.984392e+05,9.923748e+05
513,9.097134e+05,8.857537e+05
859,7.926815e+05,7.962108e+05
136,9.474908e+05,9.320310e+05


---
## Question 7 (10 Marks) (You have to explore the topic and use the equation via Numpy)
### Dont use LLMs , You can use Documentation

Implement **Multiple Linear Regression** using **only NumPy**:
- Pick random 100 datas from the dataset
- Use the Normal Equation: `θ = (XᵀX)⁻¹ Xᵀy`
- Use `Square_Footage`, `Num_Bedrooms`, and `Num_Bathrooms` as features
- Print the learned coefficients (θ values)

In [256]:
# Question 7
df = df.sample(100)
features = ['Square_Footage', 'Num_Bedrooms', 'Num_Bathrooms']


X = df[features].values
y = df['House_Price'].values

X = np.c_[np.ones((X.shape[0], 1)), X]

theta = np.linalg.inv(X.T.dot(X)).dot(X.T).dot(y)

print("Learned Coefficients (θ values):")
print(f"Intercept (θ_0)               : {theta[0]:.4f}")
print(f"Square_Footage Weight (θ_1)   : {theta[1]:.4f}")
print(f"Num_Bedrooms Weight (θ_2)     : {theta[2]:.4f}")
print(f"Num_Bathrooms Weight (θ_3)    : {theta[3]:.4f}")

Learned Coefficients (θ values):
Intercept (θ_0)               : 12712.5834
Square_Footage Weight (θ_1)   : 201.2114
Num_Bedrooms Weight (θ_2)     : 9972.2390
Num_Bathrooms Weight (θ_3)    : 7875.3869
